# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [15]:
query_phieumuon = """SELECT PMS.ID_phieu_muon, PMS.ID_ban_doc, PMS.ID_xep_gia, PMS.Ngay_muon
                    FROM oltp.Phieu_muon_sach PMS
                        JOIN olap.DIM_Ban_doc BD ON PMS.ID_ban_doc = BD.ID_ban_doc
                        JOIN olap.DIM_Xep_gia XG ON PMS.ID_xep_gia =  XG.ID_xep_gia
                        WHERE PMS.ID_ban_doc = '21110276'"""
df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library) 
print(df_phieumuon)

   ID_phieu_muon ID_ban_doc  ID_xep_gia  Ngay_muon
0        3632098   21110276     1000196   20230208
1        3645830   21110276      312019   20230314


C:\Users\admin\AppData\Local\Temp\ipykernel_17192\3780840542.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)


## Xử lý data

In [16]:
So_luot_dung = df_phieumuon.groupby(['ID_ban_doc','ID_xep_gia', 'Ngay_muon'])['ID_phieu_muon'].count().reset_index()
So_luot_dung = So_luot_dung.rename(columns={'ID_phieu_muon': 'So_luot_dung'})
print(So_luot_dung)

  ID_ban_doc  ID_xep_gia  Ngay_muon  So_luot_dung
0   21110276      312019   20230314             1
1   21110276     1000196   20230208             1


## Load data

### [Nếu cần] Clear bảng

In [3]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.FACT_Thu_vien"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [20]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.FACT_Thu_vien (ID_ban_doc, ID_xep_gia, ID_date, So_luot_dung) 
                VALUES (?, ?, ?, ?)
                """
for index, row in So_luot_dung.iterrows():
    values = (row['ID_ban_doc'], 
              row['ID_xep_gia'],
              row['Ngay_muon'],
              row['So_luot_dung'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()